# Mini Post-Training Notebook for Math Reasoning



Setup: if the required packages are not installed, run `pip install "torch" "transformers>=4.51.0" "datasets" "tqdm" "pandas" "numpy" "matplotlib"` before executing the notebook.


## Setup

Load the base model directly and keep the prompt format explicit because this is a base model, not an instruction-tuned chat model.


In [ ]:
import copy
import gc
import json
import math
import os
import random
import re
import time
from collections import Counter
from fractions import Fraction
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


In [ ]:
SEED = 1234
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
ARTIFACT_DIR = Path("artifacts")
CKPT_DIR = ARTIFACT_DIR / "checkpoints"

N_RL_CANDIDATE = 400
SFT_TRAIN_LIMIT = 500
SFT_EPOCHS = 1
SFT_MAX_STEPS = None
RL_STEPS = 25
EVAL_LIMIT = 80
EVAL_K = 3
RL_SCORE_K = 4
RL_PROMPT_BATCH_SIZE = 4
RL_GROUP_SIZE = 4

MAX_LEN = 512
EVAL_MAX_NEW_TOKENS = 256
RL_MAX_NEW_TOKENS = 128
GEN_BATCH_SIZE = 128
MAX_RESPONSE_WORDS = 160
MICRO_BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE

ARTIFACT_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError("This notebook is configured for one CUDA GPU, such as an L40 48GB.")


In [ ]:
def load_model(model_or_path):
    """Load a full model on CUDA with training-friendly cache settings."""
    model = AutoModelForCausalLM.from_pretrained(
        model_or_path,
        torch_dtype=torch.bfloat16,
        device_map=None,
    )
    model.to("cuda")
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    return model


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = load_model(MODEL_NAME)


In [ ]:
def format_prompt(problem):
    """Build the explicit base-model prompt for one math problem."""
    return (
        "Solve the math problem. Show a short solution.\n"
        "End with exactly: Final answer: <answer>\n\n"
        f"Problem:\n{problem}\n\n"
    )


def format_sft_text(problem, target_response):
    """Return the prompt and prompt-plus-target string used for SFT."""
    prompt = format_prompt(problem)
    target = target_response.strip() + tokenizer.eos_token
    return prompt, prompt + target


In [ ]:
STRICT_FINAL_RE = re.compile(r"Final answer:\s*([^\n]+)", re.IGNORECASE)
LOOSE_FINAL_PATTERNS = [
    STRICT_FINAL_RE,
    re.compile(r"(?:final answer|answer)\s+is\s*(?:\\\(?\\boxed\{)?\$?([^\n}.]+)", re.IGNORECASE),
    re.compile(r"\\boxed\{\$?([^}]+)\}", re.IGNORECASE),
]
NUMBER_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?(?:/\d[\d,]*)?")


In [ ]:
def _clean_extracted_answer(ans):
    """Strip punctuation around an extracted answer span."""
    ans = (ans or "").strip()
    ans = re.sub(r"^[*: \t]+", "", ans)
    ans = re.sub(r"[.;,\s]+$", "", ans)
    return ans if ans else None


def extract_strict_final_answer(text):
    """Extract an answer only when there is exactly one strict Final answer line."""
    matches = STRICT_FINAL_RE.findall(text or "")
    if len(matches) != 1:
        return None
    return _clean_extracted_answer(matches[0])


def extract_final_answer(text):
    """Extract an answer using strict format first, then a few loose model-output patterns."""
    strict = extract_strict_final_answer(text)
    if strict is not None:
        return strict
    candidates = []
    for pattern in LOOSE_FINAL_PATTERNS[1:]:
        candidates.extend(pattern.findall(text or ""))
    candidates = [_clean_extracted_answer(x) for x in candidates]
    candidates = [x for x in candidates if x]
    return candidates[-1] if candidates else None


def _to_fraction(num_text):
    """Convert one numeric string to a Fraction for exact comparison."""
    s = num_text.replace(",", "").strip()
    if "/" in s:
        a, b = s.split("/", 1)
        return Fraction(int(a), int(b))
    return Fraction(s)


In [ ]:
def answer_values(ans):
    """Map answer text to numeric values, including percent and decimal equivalents."""
    if ans is None:
        return set()
    raw = str(ans).strip().lower()
    percent_like = bool(re.search(r"%|percent", raw))
    raw = raw.replace("$", " ")
    raw = re.sub(r"\b(dollars?|items?|hours?|minutes?|miles?|of the original)\b", " ", raw)
    nums = NUMBER_RE.findall(raw)
    if not nums:
        return set()
    try:
        val = _to_fraction(nums[0])
    except Exception:
        return set()
    vals = {val}
    if percent_like:
        vals.add(val / 100)
    return vals


def normalize_answer(ans):
    """Return one normalized numeric value for display or debugging."""
    vals = answer_values(ans)
    if not vals:
        return None
    return sorted(vals, key=lambda x: abs(float(x)))[-1]


def equivalent(pred, gold, tol=1e-6):
    """Check numeric equivalence between a predicted answer and a gold answer."""
    pred_vals = answer_values(pred)
    gold_vals = answer_values(gold)
    if not pred_vals or not gold_vals:
        return False
    return any(abs(float(p - g)) <= tol for p in pred_vals for g in gold_vals)


In [ ]:
def has_exactly_one_final_answer(text):
    """Check strict final-answer format compliance."""
    return extract_strict_final_answer(text) is not None


def count_words(text):
    """Count whitespace-separated words in generated text."""
    return len(re.findall(r"\S+", text or ""))


def is_malformed_response(text):
    """Detect targets without exactly one strict Final answer line."""
    return extract_strict_final_answer(text) is None


def is_too_verbose(text, max_words=160):
    """Detect target responses that exceed the desired short-solution length."""
    return count_words(text) > max_words


In [ ]:
@torch.no_grad()
def generate_batch(model, prompts, temperature=0.0, max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=GEN_BATCH_SIZE):
    """Generate completions for many prompts using batched decoding."""
    tokenizer.padding_side = "left"
    do_sample = temperature is not None and temperature > 0
    completions = []
    model.eval()
    previous_use_cache = model.config.use_cache
    model.config.use_cache = True  # KV cache is important for fast autoregressive decoding.
    try:
        for start in range(0, len(prompts), batch_size):
            chunk = prompts[start:start + batch_size]
            inputs = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
            prompt_width = inputs["input_ids"].shape[1]
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature if do_sample else None,
                top_p=0.95 if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            for seq in output:
                completions.append(tokenizer.decode(seq[prompt_width:], skip_special_tokens=True))
    finally:
        model.config.use_cache = previous_use_cache
    return completions


def generate_one(model, prompt, temperature=0.0, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Generate one completion for one prompt."""
    return generate_batch(model, [prompt], temperature=temperature, max_new_tokens=max_new_tokens)[0]


def generate_k(model, prompt, k=EVAL_K, temperature=0.7, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Generate k sampled completions for one prompt."""
    return generate_batch(model, [prompt] * k, temperature=temperature, max_new_tokens=max_new_tokens)


In [ ]:
def majority_vote_answer(completions):
    """Return the most common normalized answer among sampled completions."""
    answers = [extract_final_answer(x) for x in completions]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None
    keys = []
    for ans in answers:
        vals = answer_values(ans)
        keys.append(str(sorted(vals, key=str)[0]) if vals else ans.strip().lower())
    key = Counter(keys).most_common(1)[0][0]
    for ans, k in zip(answers, keys):
        if k == key:
            return ans
    return answers[0]


def evaluate_model(model, examples, model_name, split_name, k=EVAL_K, max_new_tokens=EVAL_MAX_NEW_TOKENS, max_examples=None, batch_size=GEN_BATCH_SIZE):
    """Evaluate greedy and sampled correctness metrics on a split."""
    eval_examples = list(examples)
    if max_examples is not None:
        eval_examples = eval_examples[:max_examples]
    prompts = [format_prompt(ex["problem"]) for ex in eval_examples]
    greedy_outputs = generate_batch(model, prompts, temperature=0.0, max_new_tokens=max_new_tokens, batch_size=batch_size)
    if k == 1:
        grouped_samples = [[x] for x in greedy_outputs]
    elif k > 1:
        sample_prompts = [p for p in prompts for _ in range(k)]
        flat_samples = generate_batch(model, sample_prompts, temperature=0.7, max_new_tokens=max_new_tokens, batch_size=batch_size)
        grouped_samples = [flat_samples[i:i + k] for i in range(0, len(flat_samples), k)]
    else:
        grouped_samples = [[] for _ in eval_examples]

    rows = []
    for ex, greedy, samples in tqdm(list(zip(eval_examples, greedy_outputs, grouped_samples)), desc=f"score {model_name}/{split_name}"):
        sample_correct = [equivalent(extract_final_answer(s), ex["gold_answer"]) for s in samples]
        rows.append({
            "model": model_name,
            "split": split_name,
            "id": ex.get("id"),
            "family": ex.get("family"),
            "gold_answer": ex["gold_answer"],
            "greedy": greedy,
            "exact_at_1": float(equivalent(extract_final_answer(greedy), ex["gold_answer"])),
            "avg_at_k": float(np.mean(sample_correct)) if sample_correct else np.nan,
            "pass_at_k": float(any(sample_correct)) if sample_correct else np.nan,
            "maj_at_k": float(equivalent(majority_vote_answer(samples), ex["gold_answer"])) if sample_correct else np.nan,
            "format_compliance": float(has_exactly_one_final_answer(greedy)),
            "avg_output_length": float(np.mean([count_words(greedy)] + [count_words(s) for s in samples])),
        })

    df = pd.DataFrame(rows)
    agg_spec = {
        "pass@1": ("exact_at_1", "mean"),
        f"pass@1 [{k}]": ("avg_at_k", "mean"),
        f"pass@{k}": ("pass_at_k", "mean"),
        "format": ("format_compliance", "mean"),
        "avg_len": ("avg_output_length", "mean"),
    }
    if k >= 3:
        agg_spec[f"maj@{k}"] = ("maj_at_k", "mean")
    summary = df.groupby(["model", "split"], as_index=False).agg(**agg_spec)
    return summary, df


## Load Prepared Data

The notebook starts from prepared dirty SFT data and held-out evaluation sets.


In [ ]:
def write_jsonl(path, rows):
    """Write records as JSON lines."""
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path):
    """Read records from a JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


In [ ]:
required_artifacts = {
    "dirty_sft_pool": ARTIFACT_DIR / "synthetic_dirty_sft.jsonl",
    "id_holdout": ARTIFACT_DIR / "id_holdout.jsonl",
    "rl_candidate_pool": ARTIFACT_DIR / "rl_candidate_pool.jsonl",
}
optional_artifacts = {
    "paraphrase_holdout": ARTIFACT_DIR / "paraphrase_holdout.jsonl",
}

missing = [str(path) for path in required_artifacts.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prepared dataset artifacts. Run: python scripts/prepare_synthetic_math_data.py --difficulty hard\n"
        + "\n".join(missing)
    )


In [ ]:
dirty_sft_pool = read_jsonl(required_artifacts["dirty_sft_pool"])
id_holdout = read_jsonl(required_artifacts["id_holdout"])
rl_candidate_pool = read_jsonl(required_artifacts["rl_candidate_pool"])
paraphrase_holdout = read_jsonl(optional_artifacts["paraphrase_holdout"]) if optional_artifacts["paraphrase_holdout"].exists() else []

metadata_path = ARTIFACT_DIR / "synthetic_data_metadata.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    print("prepared data difficulty:", metadata.get("difficulty"))
    print("prepared data counts:", metadata.get("counts"))

print("dirty examples:", len(dirty_sft_pool))
print("id holdout:", len(id_holdout))
print("optional paraphrase holdout:", len(paraphrase_holdout))
print("rl candidate prompts:", len(rl_candidate_pool))
print("dirty mix:", Counter(x["corruption_type"] for x in dirty_sft_pool))
display(pd.DataFrame(dirty_sft_pool[:8])[["family", "corruption_type", "problem", "gold_answer", "target_response"]])


## SFT Filtering

A basic filter removes wrong answers, malformed outputs, answer-only targets, and long boilerplate.


Exercise candidate: adjust `keep_sft_example` to add or remove one filtering rule, then compare kept count and eval results.


In [ ]:
def canonical_problem(problem):
    """Canonicalize problem text for exact overlap checks."""
    text = (problem or "").lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9.%$: /-]+", "", text)
    return text.strip()


def is_too_short_response(text, min_words=8):
    """Detect answer-only or nearly answer-only targets."""
    return count_words(text) < min_words


def is_holdout_contamination(example, holdout_index):
    """Check whether an example exactly overlaps a held-out problem."""
    return canonical_problem(example["problem"]) in holdout_index


## Exercise 1 below

In [ ]:
def keep_sft_example(example, max_words=160, min_words=8):
    """Return whether to keep one dirty SFT example and the reason."""
    ## TODO Implement
    ## Returns True or False
    ## False if there is no strict final answer in response
    ## False if the final answer response is not equivalent to gold_answer
    ## False if response too short, < min_words
    ## False if response too long, > max_words
    ## See helper functions equivalent, is_too_short_response, is_too_verbose
    response = example["target_response"]
    pred = extract_strict_final_answer(response)

    keep = # boolean to compute
    
    return keep, "kept"


def filter_sft_pool(pool, max_words=160, min_words=8):
    """Apply the basic SFT filter and collect removal statistics."""
    stats = Counter()
    kept = []
    for ex in pool:
        ok, reason = keep_sft_example(ex, max_words=max_words, min_words=min_words)
        stats[reason] += 1
        if ok:
            kept.append(ex)
    return kept, stats


In [ ]:
sft_filtered, filter_stats = filter_sft_pool(
    dirty_sft_pool,
    max_words=MAX_RESPONSE_WORDS,
    min_words=8,
)

print("dirty examples:", len(dirty_sft_pool))
print("kept examples:", len(sft_filtered))
for key in ["removed_incorrect", "removed_malformed", "removed_too_short", "removed_verbose"]:
    print(f"{key}: {filter_stats.get(key, 0)}")


In [ ]:
def no_exact_problem_overlap(a, b):
    """Check that two example lists have no exact canonical problem overlap."""
    return not ({canonical_problem(x["problem"]) for x in a} & {canonical_problem(x["problem"]) for x in b})


In [ ]:
holdout_rows_for_leakage = id_holdout + (paraphrase_holdout if paraphrase_holdout else [])
holdout_canon_for_leakage = {canonical_problem(x["problem"]) for x in holdout_rows_for_leakage}
before_leakage_guard = len(sft_filtered)
sft_filtered = [x for x in sft_filtered if canonical_problem(x["problem"]) not in holdout_canon_for_leakage]
print("removed exact holdout overlaps:", before_leakage_guard - len(sft_filtered))

rng = random.Random(SEED + 70)
rng.shuffle(sft_filtered)
val_size = max(32, int(0.10 * len(sft_filtered)))
sft_val = sft_filtered[:val_size]
sft_train = sft_filtered[val_size:]
sft_train = sft_train[:SFT_TRAIN_LIMIT]

assert no_exact_problem_overlap(sft_train, id_holdout)
if paraphrase_holdout:
    assert no_exact_problem_overlap(sft_train, paraphrase_holdout)

write_jsonl(ARTIFACT_DIR / "sft_train_filtered.jsonl", sft_train)
write_jsonl(ARTIFACT_DIR / "id_holdout.jsonl", id_holdout)
if paraphrase_holdout:
    write_jsonl(ARTIFACT_DIR / "paraphrase_holdout.jsonl", paraphrase_holdout)
write_jsonl(ARTIFACT_DIR / "rl_candidate_pool.jsonl", rl_candidate_pool)

len(sft_train), len(sft_val)


## Baseline Evaluation

`pass@1` is greedy correctness, i.e. sampled at temperature 0.0. `pass@1 [k]` averages correctness over k sampled completions; `pass@k` checks whether any sampled completion is correct.


In [ ]:
base_summary, base_detail_df = evaluate_model(
    model,
    id_holdout,
    "base",
    "id",
    k=EVAL_K,
    max_new_tokens=EVAL_MAX_NEW_TOKENS,
    max_examples=EVAL_LIMIT,
)
display(base_summary)


## Full SFT

Manual PyTorch SFT masks prompt and padding tokens so only the target response is learned.


Exercise candidate: implement or inspect `build_labels`; prompt and padding tokens must be masked with `-100`.


In [ ]:
class SFTDataset(Dataset):
    """Thin Dataset wrapper around filtered SFT rows."""
    def __init__(self, rows):
        self.rows = list(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


In [ ]:
def build_labels(input_ids, prompt_lengths, pad_token_id):
    """Mask prompt and padding tokens so only target tokens contribute to loss."""
    labels = input_ids.clone()
    for i, plen in enumerate(prompt_lengths):
        labels[i, :plen] = -100  # Do not train on prompt tokens.
    labels[input_ids == pad_token_id] = -100
    return labels


def tokenize_sft_batch(batch):
    """Tokenize SFT examples and build masked language-model labels."""
    tokenizer.padding_side = "right"
    prompts = []
    full_texts = []
    for ex in batch:
        prompt, full = format_sft_text(ex["problem"], ex["target_response"])
        prompts.append(prompt)
        full_texts.append(full)
    prompt_lengths = [len(tokenizer(p, add_special_tokens=False)["input_ids"]) for p in prompts]
    enc = tokenizer(
        full_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_tensors="pt",
    )
    prompt_lengths = [min(x, enc["input_ids"].shape[1]) for x in prompt_lengths]
    enc["labels"] = build_labels(enc["input_ids"], prompt_lengths, tokenizer.pad_token_id)
    enc["prompt_lengths"] = torch.tensor(prompt_lengths)
    return enc


def validation_loss(model, rows, batch_size=MICRO_BATCH_SIZE):
    """Compute average SFT validation loss without updating weights."""
    loader = DataLoader(SFTDataset(rows), batch_size=batch_size, shuffle=False, collate_fn=tokenize_sft_batch)
    losses = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc="val loss"):
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            losses.append(float(outputs.loss.detach().cpu()))
    return float(np.mean(losses))


## Exercise 2

Exercise candidate: trace one SFT update step, including gradient accumulation and checkpoint evaluation.


In [ ]:
def train_sft(model, train_rows, val_rows):
    """Run a manual full-parameter SFT loop and evaluate saved checkpoints."""
    train_loader = DataLoader(
        SFTDataset(train_rows),
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        collate_fn=tokenize_sft_batch,
        drop_last=True,
    )
    total_update_steps = max(1, (len(train_loader) * SFT_EPOCHS) // GRADIENT_ACCUMULATION_STEPS)
    if SFT_MAX_STEPS is not None:
        total_update_steps = min(total_update_steps, SFT_MAX_STEPS)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    warmup_steps = max(1, int(0.03 * total_update_steps))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
    save_steps = sorted({max(1, total_update_steps // 2), total_update_steps})
    checkpoint_rows = []
    optimizer.zero_grad(set_to_none=True)
    global_step = 0
    accum = 0
    stop = False

    for epoch in range(SFT_EPOCHS):
        for batch in tqdm(train_loader, desc=f"sft epoch {epoch+1}/{SFT_EPOCHS}"):
            model.train()
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            accum += 1
            if accum % GRADIENT_ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                if global_step in save_steps:
                    ckpt_path = CKPT_DIR / f"sft_step_{global_step}"
                    model.save_pretrained(ckpt_path)
                    tokenizer.save_pretrained(ckpt_path)
                    val = validation_loss(model, val_rows)
                    id_summary, _ = evaluate_model(model, id_holdout, f"sft_step_{global_step}", "id", k=EVAL_K, max_new_tokens=EVAL_MAX_NEW_TOKENS, max_examples=EVAL_LIMIT)
                    row = {"checkpoint": str(ckpt_path), "step": global_step, "val_loss": val}
                    row["id_pass@1"] = float(id_summary["pass@1"].iloc[0])
                    row[f"id_pass@{EVAL_K}"] = float(id_summary[f"pass@{EVAL_K}"].iloc[0])
                    row["id_format"] = float(id_summary["format"].iloc[0])
                    row["id_avg_len"] = float(id_summary["avg_len"].iloc[0])
                    checkpoint_rows.append(row)
                    display(pd.DataFrame(checkpoint_rows))
                if global_step >= total_update_steps:
                    stop = True
                    break
        if stop:
            break

    final_path = CKPT_DIR / "sft_final"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    return pd.DataFrame(checkpoint_rows), final_path


In [ ]:
sft_checkpoint_table, sft_final_path = train_sft(model, sft_train, sft_val)
display(sft_checkpoint_table)


In [ ]:
def choose_checkpoint(checkpoint_table):
    """Select an SFT checkpoint using format, sampled pass rate, and length."""
    table = checkpoint_table.copy()
    if table.empty:
        return str(sft_final_path)
    table["len_penalty"] = np.maximum(0, table["id_avg_len"] - 140) / 140
    table["score"] = 0.45 * table["id_format"] + 0.45 * table[f"id_pass@{EVAL_K}"] - 0.10 * table["len_penalty"]
    chosen = table.sort_values(["score", "val_loss"], ascending=[False, True]).iloc[0]
    print("selected for RL:", chosen["checkpoint"])
    display(table.sort_values("score", ascending=False))
    return str(chosen["checkpoint"])


In [ ]:
SFT_CKPT_FOR_RL = choose_checkpoint(sft_checkpoint_table)


## RL Prompt Set

Use prompt-only data, selected near the current model frontier by sampled pass rate.


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
model = load_model(SFT_CKPT_FOR_RL)


## Exercise 3: implement `reward_fn`.


In [ ]:
def reward_fn(problem, completion, gold_answer):
    """Return exact-answer reward for one generated completion."""
    ## TODO Implement
    ## Returns 1.0 if exact-answer is equivalent to gold_answer
    ## See helpers extract_final_answer, equivalent

    reward = # 0.0 or 1.0 to compute
    
    return reward


def auxiliary_format_reward(completion):
    """Return a small optional reward for strict final-answer format."""
    return 0.05 if has_exactly_one_final_answer(completion) else 0.0


def length_penalty(completion):
    """Return a small optional penalty for overly long completions."""
    return -0.05 if count_words(completion) > 160 else 0.0


In [ ]:
def score_rl_candidates(model, candidate_pool, K=RL_SCORE_K, max_new_tokens=RL_MAX_NEW_TOKENS, batch_size=GEN_BATCH_SIZE):
    """Estimate sampled correctness for candidate RL prompts."""
    sft_train_canon = {canonical_problem(x["problem"]) for x in sft_train}
    holdout_rows = id_holdout + (paraphrase_holdout if paraphrase_holdout else [])
    holdout_index = {canonical_problem(x["problem"]) for x in holdout_rows}
    filtered = []
    for ex in candidate_pool[:N_RL_CANDIDATE]:
        if canonical_problem(ex["problem"]) in sft_train_canon:
            continue
        if is_holdout_contamination(ex, holdout_index):
            continue
        filtered.append(ex)

    prompts = [format_prompt(ex["problem"]) for ex in filtered]
    sample_prompts = [p for p in prompts for _ in range(K)]
    flat_attempts = generate_batch(model, sample_prompts, temperature=0.7, max_new_tokens=max_new_tokens, batch_size=batch_size)
    greedy_outputs = generate_batch(model, prompts, temperature=0.0, max_new_tokens=max_new_tokens, batch_size=batch_size)

    scored = []
    for i, ex in enumerate(tqdm(filtered, desc="score RL candidates")):
        attempts = flat_attempts[i * K:(i + 1) * K]
        correct = [reward_fn(ex["problem"], a, ex["gold_answer"]) for a in attempts]
        greedy = greedy_outputs[i]
        scored.append({
            **ex,
            "attempts": attempts,
            "p_hat": float(np.mean(correct)),
            "pass_k": int(any(correct)),
            "greedy_correct": int(reward_fn(ex["problem"], greedy, ex["gold_answer"])),
            "prompt_len": count_words(ex["problem"]),
        })
    return scored


def select_rl_prompt_set(scored_candidates, min_p=0.20, max_p=0.80, max_per_family=10, max_total=80, min_total=16):
    """Select diverse prompts near the model's current success frontier."""
    def make_frontier(lo, require_non_greedy=True):
        return [
            x for x in scored_candidates
            if lo < x["p_hat"] < max_p
            and x["pass_k"]
            and (not require_non_greedy or not x["greedy_correct"])
            and x["prompt_len"] <= 80
        ]

    frontier = make_frontier(min_p, require_non_greedy=True)
    if len(frontier) < min_total:
        print("RL frontier too small; broadening to p_hat > 0 with non-greedy prompts.")
        frontier = make_frontier(0.0, require_non_greedy=True)
    if len(frontier) < min_total:
        print("RL frontier still small; allowing greedy-correct prompts as fallback.")
        frontier = make_frontier(0.0, require_non_greedy=False)

    frontier.sort(key=lambda x: (x["p_hat"], x["family"]))
    per_family = Counter()
    selected = []
    for ex in frontier:
        if per_family[ex["family"]] >= max_per_family:
            continue
        selected.append({k: ex[k] for k in ["id", "problem", "gold_answer", "family", "difficulty", "template_id", "p_hat", "pass_k"]})
        per_family[ex["family"]] += 1
        if len(selected) >= max_total:
            break
    return selected


In [ ]:
scored_rl_candidates = score_rl_candidates(model, rl_candidate_pool, K=RL_SCORE_K)
rl_prompt_set = select_rl_prompt_set(
    scored_rl_candidates,
    min_p=0.20,
    max_p=0.80,
    max_per_family=10,
    max_total=80,
)
write_jsonl(ARTIFACT_DIR / "rl_prompt_set.jsonl", rl_prompt_set)

print("candidate prompts:", len(scored_rl_candidates))
print("selected RL prompts:", len(rl_prompt_set))
print("mean p_hat:", np.mean([x["p_hat"] for x in rl_prompt_set]) if rl_prompt_set else None)
print("family distribution:", Counter(x["family"] for x in rl_prompt_set))
print("difficulty distribution:", Counter(x["difficulty"] for x in rl_prompt_set))


## Tiny RL Loop

This is a GRPO-lite / REINFORCE loop with group baselines, a frozen SFT reference model, exact-answer rewards, and a small KL penalty.


## Exercise 4: inspect the group-baseline advantages and policy-gradient loss in the tiny RL loop.


In [ ]:
def build_generation_batch(prompts, G, max_new_tokens=128, temperature=0.7):
    """Generate grouped completions and return tensors needed for RL logprobs."""
    tokenizer.padding_side = "left"
    repeated = [p for p in prompts for _ in range(G)]
    enc = tokenizer(repeated, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
    prompt_width = enc["input_ids"].shape[1]
    model.eval()
    previous_use_cache = model.config.use_cache
    model.config.use_cache = True
    try:
        with torch.no_grad():
            sequences = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
    finally:
        model.config.use_cache = previous_use_cache
    attention_mask = torch.ones_like(sequences)
    attention_mask[:, :prompt_width] = enc["attention_mask"]
    completions = [tokenizer.decode(seq[prompt_width:], skip_special_tokens=True) for seq in sequences]
    prompt_lengths = torch.full((sequences.shape[0],), prompt_width, dtype=torch.long, device=device)
    return sequences, attention_mask, prompt_lengths, completions


def compute_completion_logprobs(model, sequences, attention_mask, prompt_lengths):
    """Average token log-probability over generated completion tokens only."""
    outputs = model(input_ids=sequences, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    labels = sequences[:, 1:]
    token_logprobs = F.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    positions = torch.arange(1, sequences.shape[1], device=sequences.device).unsqueeze(0)
    completion_mask = (positions >= prompt_lengths.unsqueeze(1)) & attention_mask[:, 1:].bool()
    token_counts = completion_mask.sum(dim=1).clamp_min(1)
    return (token_logprobs * completion_mask).sum(dim=1) / token_counts


In [ ]:
def train_tiny_rl(model, rl_prompts, steps=RL_STEPS, prompt_batch_size=RL_PROMPT_BATCH_SIZE, G=RL_GROUP_SIZE, lr=5e-6, beta=0.01, max_new_tokens=RL_MAX_NEW_TOKENS):
    """Run a transparent policy-gradient loop with group-normalized rewards."""
    if not rl_prompts:
        raise RuntimeError("No RL prompts selected. Broaden min_p/max_p or reduce filtering.")
    ref_model = copy.deepcopy(model)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    logs = []
    rng = random.Random(SEED + 80)

    for step in tqdm(range(1, steps + 1), desc="tiny RL"):
        batch = [rng.choice(rl_prompts) for _ in range(prompt_batch_size)]
        prompts = [format_prompt(x["problem"]) for x in batch]
        sequences, attention_mask, prompt_lengths, completions = build_generation_batch(
            prompts, G=G, max_new_tokens=max_new_tokens, temperature=0.7
        )
        rewards = []
        for ex, group in zip(batch, [completions[i:i+G] for i in range(0, len(completions), G)]):
            rewards.extend([reward_fn(ex["problem"], c, ex["gold_answer"]) for c in group])
        rewards = torch.tensor(rewards, dtype=torch.float32, device=device)

        grouped = rewards.view(prompt_batch_size, G)
        advantages = grouped - grouped.mean(dim=1, keepdim=True)
        advantages = advantages / (grouped.std(dim=1, keepdim=True) + 1e-6)
        advantages = advantages.reshape(-1)

        model.train()
        policy_logprobs = compute_completion_logprobs(model, sequences, attention_mask, prompt_lengths)
        with torch.no_grad():
            ref_logprobs = compute_completion_logprobs(ref_model, sequences, attention_mask, prompt_lengths)
        approx_kl = (policy_logprobs - ref_logprobs).mean()
        policy_loss = -(advantages.detach() * policy_logprobs).mean()
        loss = policy_loss + beta * approx_kl

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        row = {
            "step": step,
            "reward_mean": float(rewards.mean().detach().cpu()),
            "reward_std": float(rewards.std().detach().cpu()),
            "policy_loss": float(policy_loss.detach().cpu()),
            "approx_kl": float(approx_kl.detach().cpu()),
            "avg_len": float(np.mean([count_words(c) for c in completions])),
        }
        logs.append(row)
        if step % 10 == 0 or step == 1:
            print(row)
        if step == steps:
            path = CKPT_DIR / f"rl_step_{step}"
            model.save_pretrained(path)
            tokenizer.save_pretrained(path)

    del ref_model
    gc.collect()
    torch.cuda.empty_cache()
    final_path = CKPT_DIR / "rl_final"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    return pd.DataFrame(logs), final_path


In [ ]:
rl_log, RL_FINAL_PATH = train_tiny_rl(
    model,
    rl_prompt_set,
    steps=RL_STEPS,
    prompt_batch_size=RL_PROMPT_BATCH_SIZE,
    G=RL_GROUP_SIZE,
    lr=5e-6,
    beta=0.01,
    max_new_tokens=RL_MAX_NEW_TOKENS,
)
display(rl_log.tail())
rl_log.plot(x="step", y=["reward_mean", "approx_kl"], figsize=(7, 3))
plt.show()


## Final Evaluation

Evaluate base, SFT, and RL on the required in-distribution holdout.


In [ ]:
def eval_checkpoint(model_label, model_or_path, splits, k=EVAL_K):
    """Load a checkpoint, evaluate it on one or more splits, then release memory."""
    local_model = load_model(model_or_path)
    rows = []
    details = []
    for split_name, split_rows in splits:
        summary, detail = evaluate_model(local_model, split_rows, model_label, split_name, k=k, max_new_tokens=EVAL_MAX_NEW_TOKENS, max_examples=EVAL_LIMIT)
        rows.append(summary)
        details.append(detail)
    del local_model
    gc.collect()
    torch.cuda.empty_cache()
    return pd.concat(rows, ignore_index=True), pd.concat(details, ignore_index=True)


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

required_splits = [("id", id_holdout)]
final_tables = []
final_details = []

for label, path in [
    ("base", MODEL_NAME),
    ("sft", SFT_CKPT_FOR_RL),
    ("rl", RL_FINAL_PATH),
]:
    summary, details = eval_checkpoint(label, path, required_splits, k=EVAL_K)
    final_tables.append(summary)
    final_details.append(details)

final_table = pd.concat(final_tables, ignore_index=True)
final_detail_df = pd.concat(final_details, ignore_index=True)
display(final_table)
final_table.to_csv(ARTIFACT_DIR / "eval_results.csv", index=False)


## Open Ended Improvement Suggestions

Try improving SFT by increasing `SFT_TRAIN_LIMIT` or `SFT_EPOCHS`. Try improving RL by changing `N_RL_CANDIDATE`, `RL_SCORE_K`, `RL_STEPS`, `RL_GROUP_SIZE`, `beta`, or the `min_p`/`max_p` prompt frontier. Choosing the SFT checkpoint before the RL could also make a difference in final performance. Inspect examples where models differ and decide whether the reward or data should change.


## Optional Final Evals

Run this only after the main pipeline if you want to check paraphrase or GSM8K-easy transfer.


In [ ]:
def parse_gsm8k_final(answer_text):
    """Parse the final GSM8K answer after the #### marker."""
    m = re.search(r"####\s*([^\n]+)", answer_text or "")
    if not m:
        return None
    return m.group(1).strip()


def load_gsm8k_easy(n=10):
    """Load a small GSM8K-easy subset, falling back to the local JSONL file."""
    try:
        from datasets import load_dataset
        try:
            ds = load_dataset("openai/gsm8k", "main", split="test")
        except Exception:
            ds = load_dataset("gsm8k", "main", split="test")
        rows = []
        for i, ex in enumerate(ds):
            gold = parse_gsm8k_final(ex.get("answer", ""))
            if gold is None or count_words(ex["question"]) > 90:
                continue
            rows.append({
                "id": f"gsm8k_{i}",
                "family": "gsm8k_easy",
                "problem": ex["question"],
                "gold_answer": gold,
                "difficulty": 3,
                "template_id": "external_gsm8k",
            })
            if len(rows) >= n:
                break
        print("loaded GSM8K via datasets")
        return rows
    except Exception as e:
        path = ARTIFACT_DIR / "gsm8k_easy_50.jsonl"
        print(f"using local fallback {path}: {type(e).__name__}: {e}")
        return read_jsonl(path)[:n]


In [ ]:
RUN_OPTIONAL_EVALS = False

if RUN_OPTIONAL_EVALS:
    optional_splits = []
    if paraphrase_holdout:
        optional_splits.append(("paraphrase", paraphrase_holdout))
    optional_splits.append(("gsm8k_easy", load_gsm8k_easy(10)))
    optional_tables = []
    for label, path in [
        ("base", MODEL_NAME),
        ("sft", SFT_CKPT_FOR_RL),
        ("rl", RL_FINAL_PATH),
    ]:
        summary, _ = eval_checkpoint(label, path, optional_splits, k=1)
        optional_tables.append(summary)
    optional_eval_table = pd.concat(optional_tables, ignore_index=True)
    display(optional_eval_table)
